# 08 — Local LLM Analysis (Phase 8A)

**Project:** DGCA Agentic AI Regulatory Intelligence & Monitoring System
**Phase:** 8A — Free local LLM integration (no paid API calls)

## 1. Project Overview

Phase 6/7's AI analysis had exactly two paths: a real OpenAI call, or a
deterministic fallback. This phase adds a third, free option -- a local LLM
served by **Ollama** -- as the new default for development, so real
(non-templated) AI analysis can be exercised and validated without spending
any money.

```
LLM_PROVIDER=local     -->  Ollama (this phase's new default)
LLM_PROVIDER=openai    -->  OpenAI API (implemented, kept available, NOT called by default)
LLM_PROVIDER=fallback  -->  deterministic template analysis (unchanged since Phase 6)
```

**No code in `pipeline/` was redesigned.** `generate_ai_analysis(row,
extracted_text, comparison, config)` -- the exact function Phase 7's
orchestrator already calls -- keeps its signature and keeps returning the
same `AIAnalysisResult`. Only its *inside* changed: it now dispatches on
`config.llm_provider` before falling through to the OpenAI-or-fallback logic
that already existed. `notebooks/07_End_to_End_Workflow.ipynb` needs no
changes at all.

**This notebook makes zero calls to the OpenAI API.** The OpenAI code path
(`pipeline.analysis._try_openai_analysis`) is untouched and still there, but
nothing in this notebook sets `LLM_PROVIDER=openai`, and Section 10 proves
that even if it *were* selected, an unconfigured key means no network call
is attempted at all.


## 2. Imports

In [ ]:
import sys
import time
from pathlib import Path

_project_root_for_import = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_project_root_for_import) not in sys.path:
    sys.path.insert(0, str(_project_root_for_import))

import requests

from pipeline.config import load_config
from pipeline.utils import configure_logging
from pipeline import storage
from pipeline import comparison
from pipeline import analysis

print("Imports OK -- pipeline package loaded.")


## 3. Provider Configuration

Loaded the same way as every other notebook. Only booleans/non-secret
fields are ever printed -- `OPENAI_API_KEY` is checked with
`config.openai_configured()`, which returns `True`/`False` only. Its actual
value is never read into this notebook.


In [ ]:
PROJECT_ROOT = _project_root_for_import
ENV_PATH = PROJECT_ROOT / "config" / ".env"
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "documents"
LOG_DIR = PROJECT_ROOT / "logs"

config = load_config(ENV_PATH)

print(f"LLM_PROVIDER     : {config.llm_provider}")
print(f"OLLAMA_BASE_URL  : {config.ollama_base_url}")
print(f"OLLAMA_MODEL     : {config.ollama_model}")
print(f"OPENAI configured (bool only, value never read): {config.openai_configured()}")
print(f"EMAIL_PROVIDER   : {config.email_provider}  |  TEST_MODE: {config.test_mode}")

assert config.llm_provider == "local", "This phase expects LLM_PROVIDER=local as the default."
print()
print("Provider confirmed: local (Ollama) -- no OpenAI calls will be made in this notebook.")


## 4. Logging

In [ ]:
logger = configure_logging(LOG_DIR, "local_llm_analysis")


## 5. Ollama Availability Check

`pipeline.analysis.check_ollama_available()` is the exact function
`generate_ai_analysis()` uses internally before ever attempting a local
call -- this section just surfaces its result explicitly.


In [ ]:
ollama_available = analysis.check_ollama_available(config.ollama_base_url)
print(f"Ollama reachable at {config.ollama_base_url}: {ollama_available}")

if not ollama_available:
    print()
    print("Ollama is not reachable. Install it from https://ollama.com and ensure the")
    print("service is running, then re-run this notebook. No local LLM tests will")
    print("succeed until this is True -- they will correctly fall back to the")
    print("deterministic analysis instead (see Section 9).")


## 6. Model Configuration

Confirms the configured model (`llama3.1:8b` by default) is actually pulled
and available -- not just that *some* model exists.


In [ ]:
if ollama_available:
    resp = requests.get(f"{config.ollama_base_url}/api/tags", timeout=10)
    resp.raise_for_status()
    models = resp.json().get("models", [])
    model_names = [m["name"] for m in models]
    print(f"Models available in Ollama: {model_names}")

    target = next((m for m in models if m["name"] == config.ollama_model), None)
    if target is None:
        print(f"WARNING: configured model not found: {config.ollama_model}. Run: ollama pull {config.ollama_model}")
    else:
        size_gb = target.get("size", 0) / (1024 ** 3)
        details = target.get("details", {})
        param_size = details.get("parameter_size", "unknown")
        quant_level = details.get("quantization_level", "unknown")
        context_len = details.get("context_length", "unknown")
        print()
        print(f"Configured model : {config.ollama_model}")
        print(f"Disk size        : {size_gb:.2f} GB")
        print(f"Parameter size   : {param_size}")
        print(f"Quantization     : {quant_level}")
        print(f"Context length   : {context_len}")
else:
    print("Skipped -- Ollama not reachable (see Section 5).")


## 7. Structured Output Schema

`AIAnalysisResult` -- unchanged since Phase 6. Every provider (local,
OpenAI, fallback) returns exactly this schema; only `analysis_mode` differs
(`"ai-local"` / `"ai-openai"` / `"fallback"` / `"degraded"`), which is how
downstream consumers (and this notebook's assertions) can tell which path
actually ran.


In [ ]:
import inspect
print(inspect.getsource(analysis.AIAnalysisResult))


## 8. Real DGCA Publication Test — local LLM, no previous version

A genuine DGCA regulation already in the Phase 3 database, with real
extracted text from Phase 4 (not synthetic). This publication has no
previous version on record (confirmed in Phase 5/7), so it also exercises
requirement 9's "first-time publication" instruction to the model.


In [ ]:
conn = storage.get_connection()

TEST_PUBLICATION_ID = "regulation-yv5UwjfEILZBCtSQvOHdA"
row = conn.execute("SELECT * FROM publications WHERE publication_id = %s", (TEST_PUBLICATION_ID,)).fetchone()
extracted_text = (DOCUMENTS_DIR / f"{TEST_PUBLICATION_ID}.txt").read_text(encoding="utf-8")

print(f"Publication : {row['title']}")
print(f"Category    : {row['category']}")
print(f"Published   : {row['published_date']}")
print(f"Text length : {len(extracted_text)} characters")

previous_row = comparison.find_previous_version(conn, TEST_PUBLICATION_ID)
print(f"Previous version on record: {previous_row is not None}")


In [ ]:
print("Calling local Ollama model -- this takes roughly a minute on an 8B model...")
t0 = time.time()
result_no_previous = analysis.generate_ai_analysis(row, extracted_text, None, config)
elapsed = time.time() - t0
print(f"Elapsed: {elapsed:.1f}s")
print()
print(result_no_previous.model_dump_json(indent=2))


## 9. Structured Output Validation

Confirms the local LLM actually produced a valid `AIAnalysisResult` (not a
silent fallback), that it used the local provider, that `key_changes`
entries carry a source-traceability quote (a parenthetical, per this
phase's prompt requirement), and that the result converts cleanly into
Phase 1's `DGCAUpdate` schema.


In [ ]:
assert result_no_previous.analysis_mode == "ai-local", (
    f"Expected the real local LLM to run, got analysis_mode={result_no_previous.analysis_mode!r} "
    "-- check Sections 5-6 for why Ollama might not be available."
)
assert result_no_previous.summary.strip() != ""
assert result_no_previous.impact_level in ("LOW", "MEDIUM", "HIGH", "CRITICAL")

traceable = [kc for kc in result_no_previous.key_changes if "(" in kc and ")" in kc]
print(f"key_changes with a parenthetical source quote: {len(traceable)}/{len(result_no_previous.key_changes)}")
assert len(result_no_previous.key_changes) == 0 or len(traceable) > 0, "Expected at least one traceable key_change."

from pipeline.email_pipeline import DGCAUpdate

update = DGCAUpdate(
    title=row["title"],
    category=row["category"],
    published_date=row["published_date"] or "Not specified in the publication.",
    summary=result_no_previous.summary,
    key_changes=result_no_previous.key_changes,
    effective_date=result_no_previous.effective_date,
    impact_level=result_no_previous.impact_level,
    impacted_departments=result_no_previous.impacted_departments,
    recommended_actions=result_no_previous.recommended_actions,
    dgca_url=row["publication_url"],
    pdf_url=row["pdf_url"],
)
print()
print("Converted to DGCAUpdate successfully:")
print(f"  title: {update.title[:70]}...")
print(f"  impact_level: {update.impact_level}")
print(f"  effective_date: {update.effective_date}")
print()
print("Structured output validation PASSED.")


## 10. Local LLM test — with a previous-version comparison available

Real DGCA data has no genuine revision pair (established in Phases 5 and
7), so requirement 9's "if a previous version exists, include the Phase 5
comparison result" is demonstrated using Phase 7's own controlled-test
fixture pair (`e2e-test-car-x-rev1`/`rev2`), which is **already sitting in
this same database** from Phase 7's end-to-end run -- not fabricated fresh
here. This proves the comparison-inclusion wiring works with the local
provider; Section 8 above is the notebook's real-DGCA-publication test.


In [ ]:
COMPARISON_PUBLICATION_ID = "e2e-test-car-x-rev2"
comp_row = conn.execute("SELECT * FROM publications WHERE publication_id = %s", (COMPARISON_PUBLICATION_ID,)).fetchone()

if comp_row is None:
    print("Phase 7 test fixture not present in this database (07_End_to_End_Workflow.ipynb hasn't been run) -- skipping this demonstration.")
    result_with_previous = None
else:
    comp_text = (DOCUMENTS_DIR / f"{COMPARISON_PUBLICATION_ID}.txt").read_text(encoding="utf-8")
    comp_previous_row = comparison.find_previous_version(conn, COMPARISON_PUBLICATION_ID)
    comp_previous_id = comp_previous_row["publication_id"] if comp_previous_row else None
    print(f"Publication: {comp_row['title']}")
    print(f"Previous version found: {comp_previous_id}")

    comparison_result = None
    if comp_previous_row is not None:
        old_text_path = DOCUMENTS_DIR / f"{comp_previous_id}.txt"
        if old_text_path.exists():
            _diff, comparison_result = comparison.compare_documents(
                old_text_path.read_text(encoding="utf-8"), comp_text, config
            )
            print(f"Comparison: {comparison_result.total_changes} change(s), level {comparison_result.overall_change_level}")

    print()
    print("Calling local Ollama model with comparison context...")
    t0 = time.time()
    result_with_previous = analysis.generate_ai_analysis(comp_row, comp_text, comparison_result, config)
    print(f"Elapsed: {time.time() - t0:.1f}s")
    print()
    print(result_with_previous.model_dump_json(indent=2))


In [ ]:
if result_with_previous is not None:
    assert result_with_previous.analysis_mode == "ai-local"
    print("Previous-version comparison correctly reached the local LLM and influenced its analysis.")
else:
    print("Skipped (see message above).")


## 11. Fallback Test — local LLM unavailable

Requirement 11. Points at a port nothing is listening on to simulate Ollama
being down, and confirms `generate_ai_analysis()` degrades to the
deterministic analysis automatically, with a logged reason -- never an
exception, never a crash.


In [ ]:
import dataclasses

broken_config = dataclasses.replace(config, ollama_base_url="http://localhost:9999")

result_fallback = analysis.generate_ai_analysis(row, extracted_text, None, broken_config)
print(result_fallback.model_dump_json(indent=2))

assert result_fallback.analysis_mode == "fallback"
print()
print("Fallback-on-unavailable test PASSED: deterministic analysis used automatically, no crash.")


## 12. Fallback Test — `LLM_PROVIDER=fallback` (explicit, by configuration)

In [ ]:
fallback_only_config = dataclasses.replace(config, llm_provider="fallback")

result_explicit_fallback = analysis.generate_ai_analysis(row, extracted_text, None, fallback_only_config)
assert result_explicit_fallback.analysis_mode == "fallback"
print("Explicit LLM_PROVIDER=fallback test PASSED.")
print(f"Same schema as the local-LLM result? {set(result_explicit_fallback.model_dump().keys()) == set(result_no_previous.model_dump().keys())}")


## 13. OpenAI path — kept intact, confirmed NOT called

Sets `LLM_PROVIDER=openai` (only for this one assertion) against the
**current, unmodified** `OPENAI_API_KEY` in `config/.env`. Since that key is
not configured as a real credential in this test run, `_try_openai_analysis()`
detects that via `config.openai_configured()` and returns the deterministic
fallback **before constructing an OpenAI client or making any network
request** -- so no API call, paid or otherwise, happens here. This does not
read, print, or alter the key in any way.


In [ ]:
openai_provider_config = dataclasses.replace(config, llm_provider="openai")

result_openai_path = analysis.generate_ai_analysis(row, extracted_text, None, openai_provider_config)
print(f"analysis_mode: {result_openai_path.analysis_mode}")
print(f"OPENAI_API_KEY configured (bool only): {config.openai_configured()}")

assert result_openai_path.analysis_mode == "fallback", "No real key configured -- must fall back, not call OpenAI."
print()
print("Confirmed: selecting the openai provider without a real key never reaches the OpenAI SDK/network layer.")
print("The OpenAI code path remains implemented and available for whenever a real key + explicit choice are both present.")


## 14. Security Verification

Automated checks for every item on this phase's security checklist.


In [ ]:
gitignore_text = (PROJECT_ROOT / ".gitignore").read_text(encoding="utf-8")
assert {".env", ".env.*"} <= set(gitignore_text.splitlines())
print("Git ignores .env and .env.*: PASSED")

import re as _re
notebook_source = Path.cwd().glob("*.ipynb")
leaked = []
for nb_path in notebook_source:
    text = nb_path.read_text(encoding="utf-8")
    if _re.search(r"sk-[A-Za-z0-9]{20,}", text):
        leaked.append(nb_path.name)
assert not leaked, f"Possible key leakage found in: {leaked}"
print("No API-key-shaped string found in any notebook file on disk: PASSED")

log_files = list(LOG_DIR.glob("*.log"))
leaked_logs = []
for lf in log_files:
    if _re.search(r"sk-[A-Za-z0-9]{20,}", lf.read_text(encoding="utf-8", errors="ignore")):
        leaked_logs.append(lf.name)
assert not leaked_logs, f"Possible key leakage found in logs: {leaked_logs}"
print("No API-key-shaped string found in any log file: PASSED")

readme_text = (PROJECT_ROOT / "README.md").read_text(encoding="utf-8")
assert not _re.search(r"sk-[A-Za-z0-9]{20,}", readme_text)
print("README.md contains no API-key-shaped string: PASSED")

print()
print("All security checks PASSED. The OPENAI_API_KEY value itself was never read by this notebook.")


## 15. Results

In [ ]:
print("=" * 70)
print("PHASE 8A TEST RESULTS")
print("=" * 70)
print(f"Ollama installed & reachable        : {ollama_available}")
print(f"Model configured                    : {config.ollama_model}")
print(f"Real DGCA publication (no previous) : {row['title'][:60]}")
print(f"  -> analysis_mode                  : {result_no_previous.analysis_mode}")
print(f"  -> impact_level                   : {result_no_previous.impact_level}")
print(f"  -> effective_date                 : {result_no_previous.effective_date}")
print(f"  -> key_changes with traceability  : {len(traceable)}/{len(result_no_previous.key_changes)}")
if result_with_previous is not None:
    print(f"Comparison-path test (Phase 7 fixture) -> analysis_mode: {result_with_previous.analysis_mode}")
print(f"Fallback-on-unavailable test         : PASSED (analysis_mode={result_fallback.analysis_mode})")
print(f"Explicit LLM_PROVIDER=fallback test  : PASSED (analysis_mode={result_explicit_fallback.analysis_mode})")
print(f"OpenAI path selected without key     : PASSED, no API call made (analysis_mode={result_openai_path.analysis_mode})")
print(f"DGCAUpdate conversion                : PASSED")
print(f"Security checks                      : PASSED")
print("=" * 70)

conn.close()


## 16. Limitations & Next Steps

### What works
- Local LLM analysis via Ollama (`llama3.1:8b`) produces genuine,
  non-templated `AIAnalysisResult` output against real DGCA text, with
  source-traceable `key_changes` and correct first-time-publication framing
  when no previous version exists.
- Automatic, logged fallback to the deterministic analysis when Ollama is
  unavailable, and when `LLM_PROVIDER=fallback` is set explicitly.
- The OpenAI path remains fully implemented but provably unreached without
  an explicit provider choice *and* a real key -- zero API spend risk.
- `pipeline/` and `notebooks/07_End_to_End_Workflow.ipynb` required no
  changes beyond `pipeline/analysis.py`'s internals and `pipeline/config.py`
  gaining three new fields -- the public `generate_ai_analysis()` signature
  is unchanged.

### Limitations
- **Latency**: a single local analysis call took roughly a minute on this
  machine's RTX 4050 -- noticeably slower than a typical OpenAI response.
  Fine for on-demand/batch use (Phase 7's orchestrator), not for anything
  needing sub-second responses.
- **No output-quality benchmarking yet** against the OpenAI path (which has
  never actually been called in this project) -- only against the
  deterministic fallback, which the LLM output is qualitatively richer than
  in every test run here, but that's not a rigorous comparison.
- `impacted_departments` are inferred by the model from context (e.g. "DGCA
  regional offices" for a CAR issuance procedure document) rather than
  always being literally named in the source text -- reasonable inference,
  but worth knowing when auditing outputs for strict traceability.
- Still only tested against a small number of documents (one real, one
  Phase 7 test fixture) -- not a large sample.

### Not started yet (by design)
- Phase 8B: Streamlit dashboard -- explicitly not started per instructions.

Stopping here per instructions -- waiting for further direction before
dashboard development.
